# Turnover rate rolling average: experiment (ticket 2011)

Scores every rolling average setup on the turnover rate (leavers over estimated employees), then re-scores the best on a temporal hide. Results go to S3 under `RESULTS_PREFIX/RUN_LABEL/`, one file per variant, so an interrupted run resumes where it stopped.

- **Test A (workplace folds):** held-out workplaces are removed from the rolling average inputs, then scored on their own known rates. Varies split, window and fill cap (the method does not affect the rolling average).
- **Test B (temporal tail hide):** each workplace's last 12 months with a known rate are hidden and the pipeline's imputation runs. Runs for the top `TEST_B_TOP_N` test A setups, each with both methods.

**Before running:** set `BRANCH` and `RESULTS_PREFIX`. **Stop the notebook instance when finished.** Writes only to `RESULTS_PREFIX`.

Aggregate error is worker-weighted, at service by CSSR by job role by date. Workplace MAE, bias and jumpiness are reported alongside.

## 0. Setup (SageMaker only)

Run the next two cells once, then restart the kernel.

In [ ]:
%pip install polars==1.44.1 pyarrow boto3

In [ ]:
!rm -rf repo
!git clone --depth 1 --branch 2011-leavers-imp https://github.com/NMDSdevopsServiceAdm/DataEngineering.git repo
!git -C repo rev-parse HEAD

## 1. Parameters

- `BRANCH`: the branch whose data bucket `sfc-<BRANCH>-datasets` holds the cleaned SLV data (read only).
- `RESULTS_PREFIX`: where results are written; `RUN_LABEL` names this run's folder.
- `VARIANT_NAMES`: a list of fill names (such as `["current_6mo_2y"]`) to run test A for, or `None` for all.
- `TEST_B_TOP_N`: how many of the best test A setups also run test B.
- `LOCAL_ROOT`: set only to dry-run against local files; `None` on SageMaker.

In [ ]:
BRANCH = "2011-leavers-imp"
DATA_BUCKET = f"sfc-{BRANCH}-datasets"
CLEANED_DATASET = "domain=03_ind_cqc/dataset=03_starters_leavers_vacancies_02_clean/"
RESULTS_PREFIX = "s3://skillsforcare-nonprod/sagemaker-notebooks/turnover-rolling-avg/"
RUN_LABEL = "run1"
VARIANT_NAMES = None
TEST_B_TOP_N = 3
N_FOLDS = 5
FOLD_SEED = 42
LOCAL_ROOT = None
REPO_DIR = "repo"

## Setup

Imports and result locations.

In [ ]:
import sys
from pathlib import Path

import polars as pl

sys.path.insert(0, str(Path(REPO_DIR).resolve()))

import experiments.rate_rolling_average.experiment_utils as x
from polars_utils import utils as polars_utils
from projects._03_independent_cqc.utils import model_evaluation_utils as evaluation

LOCAL = LOCAL_ROOT is not None
RESULTS_DIR = (
    str(Path(LOCAL_ROOT) / "results" / RUN_LABEL) if LOCAL else f"{RESULTS_PREFIX.rstrip('/')}/{RUN_LABEL}"
)
SOURCE = str(Path(LOCAL_ROOT) / "cleaned") if LOCAL else f"s3://{DATA_BUCKET}/{CLEANED_DATASET}"
PREPARED_PATH = Path("prepared_2011.parquet")
FILL_VARIANTS = {v.fill_name: v for v in x.VARIANTS if v.method == "nominal"}
SELECTED_FILLS = [v for name, v in FILL_VARIANTS.items() if VARIANT_NAMES is None or name in VARIANT_NAMES]

## 2. Prepare the data

Keeps only the columns needed, adds the workplace size band (sum of `estimate_filled_posts_by_job_role` per workplace and date) and the workplace folds, then writes the result locally and scans it back. Rows are one per workplace, job role and date. The trendline fill depends on the fill cap, so it is added per variant.

In [ ]:
source_lf = pl.scan_parquet(f"{SOURCE}/*.parquet") if LOCAL else polars_utils.scan_parquet(SOURCE)
cleaned_lf = source_lf.select(
    x.LOCATION, x.DATE, x.SERVICE, x.REGION, x.CSSR, x.ROLE, x.SIZE_POSTS, x.WORKERS, x.RATE
)
cleaned_lf = x.add_size_band(cleaned_lf).drop(x.SIZE_POSTS)
cleaned_lf = evaluation.assign_location_folds(cleaned_lf, x.LOCATION, N_FOLDS, FOLD_SEED)
cleaned_lf.sink_parquet(PREPARED_PATH)

prepared_lf = pl.scan_parquet(PREPARED_PATH)
print(prepared_lf.select(pl.len(), pl.col(x.LOCATION).n_unique()).collect())
print(prepared_lf.group_by(x.SIZE_BAND).len().sort(x.SIZE_BAND).collect())

## 3. Test A: held-out workplaces

One file per split, window and fill cap in `test_a/`, skipping any already written.

In [ ]:
def variant_columns(scores: dict, variant: x.Variant) -> pl.DataFrame:
    return pl.DataFrame(
        {
            "variant": variant.name,
            "fill_name": variant.fill_name,
            "split": variant.split,
            "window_months": variant.window_months,
            "fill_cap": variant.fill_cap,
            "method": variant.method,
            **scores,
        }
    )


finished = set(x.list_results("test_a/", RESULTS_DIR, LOCAL))
for variant in SELECTED_FILLS:
    name = f"test_a/{variant.fill_name}.parquet"
    if name in finished:
        continue
    pred_lf = x.predict_out_of_fold(prepared_lf, variant, N_FOLDS)
    scores = x.score_variant(pred_lf, x.ROLLING, x.RATE_TRUTH, pl.col(x.RATE_TRUTH).is_not_null())
    x.write_result(variant_columns(scores, variant), name, RESULTS_DIR, LOCAL)
    print(variant.fill_name, "done")

## 4. Test A summary

Lower is better for every measure except bias, where closer to 0 is better.

In [ ]:
def load_all(prefix: str) -> pl.DataFrame:
    return pl.concat(
        [x.read_result(name, RESULTS_DIR, LOCAL) for name in x.list_results(prefix, RESULTS_DIR, LOCAL)]
    )


pl.Config.set_tbl_cols(20)
pl.Config.set_tbl_rows(60)
test_a_df = load_all("test_a/").drop("method")
test_a_df.sort(x.AGGREGATE_ERROR)

## 5. Test B: temporal tail hide

Runs the top `TEST_B_TOP_N` setups by test A aggregate error, each with both methods. Each workplace's last 12 months with a known rate are hidden, the pipeline's imputation runs, and the full imputed rate is scored on the hidden values.

In [ ]:
top_fills = test_a_df.sort(x.AGGREGATE_ERROR)["fill_name"].to_list()[:TEST_B_TOP_N]
test_b_variants = [v for v in x.VARIANTS if v.fill_name in top_fills]

hidden_lf = x.hide_last_months(prepared_lf)
finished = set(x.list_results("test_b/", RESULTS_DIR, LOCAL))
for variant in test_b_variants:
    name = f"test_b/{variant.name}.parquet"
    if name in finished:
        continue
    imputed_lf = x.run_imputation_with_variant(hidden_lf, variant)
    scores = x.score_variant(imputed_lf, x.IMPUTED, x.RATE_TRUTH, pl.col(x.HIDDEN))
    x.write_result(variant_columns(scores, variant), name, RESULTS_DIR, LOCAL)
    print(variant.name, "done")

## 6. Mean-of-rates baseline

The winner of test B, re-run with every workplace counting once (mean of rates instead of leavers over employees). Set `WINNER_NAME` to the winning variant name from section 7.

In [ ]:
WINNER_NAME = None  # for example "current_6mo_2y_ratio"

if WINNER_NAME:
    winner = next(v for v in x.VARIANTS if v.name == WINNER_NAME)
    baseline_lf = x.run_imputation_with_variant(x.as_mean_of_rates(hidden_lf), winner)
    baseline_scores = x.score_variant(baseline_lf, x.IMPUTED, x.RATE_TRUTH, pl.col(x.HIDDEN))
    x.write_result(
        variant_columns(baseline_scores, winner).with_columns(pl.lit(f"{WINNER_NAME}_mean_of_rates").alias("variant")),
        f"baseline/{WINNER_NAME}_mean_of_rates.parquet",
        RESULTS_DIR,
        LOCAL,
    )

## 7. Test B summary

Same measures on the hidden rows. Jumpiness is over all rows. Open `compare.ipynb` to choose the setup.

In [ ]:
load_all("test_b/").sort(x.AGGREGATE_ERROR)